In [6]:
# Imports
import sys
from pathlib import Path
import warnings

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.pategan.models import PATEGAN

warnings.filterwarnings("ignore", message="X does not have valid feature names")

# ---------- Custom PATEGAN for CAR ----------
class PATEGANCar(PATEGAN):
    def __init__(self):
        super().__init__(
            epsilon=5.0,     # relaxed privacy → better scores
            delta=1e-5,
            num_teachers=5,  # fewer teachers → more data per teacher
            niter=8000,      # you can change to 1000/8000 later
            batch_size=64,
            learning_rate=5e-4,
            lambda_gp=5.0,
            random_state=42,
        )

# ---------------- Preprocess data (CAR) ----------------
dataset_path = ROOT / "raw_data" / "magic.csv"
output_path = ROOT / "discretized_data" / "magic.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path))

# ---------------- Run Train/Test/Synthetic pipeline ----------------
input_csv = str(output_path)          # discretised CAR data
output_dir = str(ROOT / "sample_data" / "magic")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "magic" / "pategan")

model_preview = PATEGANCar()
print("PATEGAN will train for:", model_preview.niter, "iterations")

pipeline = TrainTestSplitPipeline(model=PATEGANCar)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\magic.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\magic.csv
PATEGAN will train for: 8000 iterations
Loaded data with shape: (19020, 11)
Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
0    0.648396
1    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.648265
1    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)
Remapped y classes: [0, 1] -> [0, 1]
Loaded training data: X shape=(15216, 10), y shape=(15216,)
Training PATE-GAN with ε=5.0, δ=1e-05
Teachers: 5, Iterations: 8000


Training: 100%|██████████| 8000/8000 [02:02<00:00, 65.10it/s] 
C:\Users\Prabu\Downloads\Katabatic\katabatic\models\pategan\models.py:503: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_synth[y_col] = y_synth[y_col].astype(int)


Training completed!

Generating 15216 synthetic samples...
Remapped y_test.csv to match synthetic data encoding
Saved x_synth.csv to C:\Users\Prabu\Downloads\Katabatic\synthetic\magic\pategan\x_synth.csv
Saved y_synth.csv to C:\Users\Prabu\Downloads\Katabatic\synthetic\magic\pategan\y_synth.csv
Saved metadata.json to C:\Users\Prabu\Downloads\Katabatic\synthetic\magic\pategan\metadata.json

Results saved to: Results\magic\pategan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6966
F1 Score: 0.6116
AUC: 0.7651

MLP:
Accuracy: 0.7487
F1 Score: 0.7147
AUC: 0.7170

RF:
Accuracy: 0.7019
F1 Score: 0.6207
AUC: 0.7412

XGBoost:
Accuracy: 0.7072
F1 Score: 0.6459
AUC: 0.6621
Train test split pipeline executed successfully.


In [7]:
import pandas as pd

rows = [
    {"Model": "LR",      "Metric": "Accuracy", "Value": 0.6966},
    {"Model": "LR",      "Metric": "F1 Score", "Value": 0.6116},
    {"Model": "LR",      "Metric": "AUC",      "Value": 0.7651},

    {"Model": "MLP",     "Metric": "Accuracy", "Value": 0.7487},
    {"Model": "MLP",     "Metric": "F1 Score", "Value": 0.7147},
    {"Model": "MLP",     "Metric": "AUC",      "Value": 0.7170},

    {"Model": "RF",      "Metric": "Accuracy", "Value": 0.7019},
    {"Model": "RF",      "Metric": "F1 Score", "Value": 0.6207},
    {"Model": "RF",      "Metric": "AUC",      "Value": 0.7412},

    {"Model": "XGBoost", "Metric": "Accuracy", "Value": 0.7072},
    {"Model": "XGBoost", "Metric": "F1 Score", "Value": 0.6459},
    {"Model": "XGBoost", "Metric": "AUC",      "Value": 0.6621},
]

df = pd.DataFrame(rows)

save_path = r"C:\Users\Prabu\Downloads\petgan_magic_tstr.csv"
df.to_csv(save_path, index=False)

print("Saved CSV to:", save_path)
df


Saved CSV to: C:\Users\Prabu\Downloads\petgan_magic_tstr.csv


,Model,Metric,Value
0,LR,Accuracy,0.6966
1,LR,F1 Score,0.6116
2,LR,AUC,0.7651
3,MLP,Accuracy,0.7487
4,MLP,F1 Score,0.7147
5,MLP,AUC,0.7170
6,RF,Accuracy,0.7019
7,RF,F1 Score,0.6207
8,RF,AUC,0.7412
9,XGBoost,Accuracy,0.7072
